### Loading Libraries and Params

In [0]:
!pip install openpyxl

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
import os, json, builtins
from datetime import date
from datetime import datetime as dt
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.window import Window

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
COHORTS_DIR = f"{VOL31}/cohorts/claims"
HCP_UNIVERSE_PATH = f"{VOL31}/HCP_Universe_without_call_hcps"
QUERY_OUTPUT_31 = f"{VOL31}/hcp_level_model/ops_pipeline/query_output"
TARGET_VARIABLE_FOLDER = f"{VOL31}/notebook_output/target_variable"
OUTPUT_PATH = f"{TARGET_VARIABLE_FOLDER}/target_variable"

# Target = Kerendia NBRx written by the HCP in the cohort's prediction month (full calendar month, claims offset 0)
TARGET_COL = "KERENDIA_NBRX"

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)
assert meta["feature_family"] == "claims", meta["feature_family"]
assert len(cohorts) == meta["cohort_count"] and (cohorts["offset_days"] == 0).all()

# Prediction window = the full prediction_month (1st .. last day); the cohort file has no prediction start/end columns
df_cohort = (spark.createDataFrame(cohorts[['cohort_id','dataset','prediction_month','lookback_end_date']].astype(str))
       .withColumn("PREDICTION_START_DATE", F.to_date("prediction_month"))
       .withColumn("PREDICTION_END_DATE", F.last_day("PREDICTION_START_DATE"))
       .withColumn("LOOKBACK_END_M", F.to_date("lookback_end_date"))
       .drop("prediction_month", "lookback_end_date"))

hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()
n_hcp = hcp_universe.count()
print(f"Claims cohorts: {len(cohorts)} ({cohorts.cohort_id.iloc[0]}..{cohorts.cohort_id.iloc[-1]}), "
      f"lookback end {cohorts.lookback_end_date.iloc[0]}..{cohorts.lookback_end_date.iloc[-1]} | HCP universe: {n_hcp:,}")

In [0]:
# Prediction windows span c1..c28 prediction months; NBRx needs 365 days of history before the first window
coh_pd = df_cohort.select("PREDICTION_START_DATE", "PREDICTION_END_DATE").toPandas()
data_start = str(coh_pd.PREDICTION_START_DATE.min())                                   # 2024-07-01 (c1)
data_end = str(coh_pd.PREDICTION_END_DATE.max())                                       # 2026-10-31 (c28)
data_start_nbrx = str(pd.Timestamp(data_start) - pd.Timedelta(days=365))[:10]          # 365-day NBRx lookback
print(f"data_start_nbrx={data_start_nbrx} | data_start={data_start} | data_end={data_end}")

In [0]:
# NBA 3.1 universe flags (C1..C5 inclusion criteria); only PROVIDER_ID is used downstream
hcp_universe_flags = spark.read.parquet(HCP_UNIVERSE_PATH)
flag_cols = [c for c in hcp_universe_flags.columns if c != "PROVIDER_ID"]
hcp_universe_flags.agg(*[F.sum(c).alias(f"total_{c}") for c in flag_cols]).display()

In [0]:
# LOAD PAID KERENDIA CLAIMS (data_start_nbrx .. data_end, from the claims cohorts)
query = f'''
    SELECT DISTINCT A.PATIENT_ID, A.PROVIDER_ID, A.CLAIM_ID, A.SVC_DT, B.PRODUCT_GROUP 
    FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
    USING (NDC_CD)
    WHERE CLAIM_TYPE = 'PD'
    AND SVC_DT >= '{data_start_nbrx}' 
    AND SVC_DT <= '{data_end}'
    AND PRODUCT_GROUP = 'KERENDIA'
'''

paid_claims = Get_Data_Snowflakes(query)
print('Shape :', paid_claims.count(), len(paid_claims.columns))

In [0]:
from pyspark.sql.functions import col, min as spark_min, lag, expr, when, row_number

# Step 1: Assign rank per patient-product-date to pick the first claim
dedup_window = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP", "SVC_DT").orderBy("CLAIM_ID")

paid_claims = paid_claims.withColumn("CLAIM_RANK", row_number().over(dedup_window))

# Step 2: Define a window for NBRx calculation (per patient-product history)
window_spec = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP").orderBy("SVC_DT")

# Compute the first-ever claim date per patient-product
paid_claims = paid_claims.withColumn("MIN_SVC_DT", spark_min("SVC_DT").over(window_spec))

# Compute the lagged service date (previous claim date for that product)
paid_claims = paid_claims.withColumn("LAG_SVC_DT", lag("SVC_DT").over(window_spec))

# Step 3: Calculate NBRX_FLAG only for the first claim per day
paid_claims_w_nbrx = paid_claims.withColumn(
    "NBRX_FLAG",
    when((col("MIN_SVC_DT") == col("SVC_DT")) & (col("CLAIM_RANK") == 1), 1)
    .when((col("LAG_SVC_DT").isNotNull()) & (col("LAG_SVC_DT") < expr("date_add(SVC_DT, -365)")) & (col("CLAIM_RANK") == 1), 1)
    .otherwise(0)
)

# Step 4: Drop intermediate columns
paid_claims_w_nbrx = paid_claims_w_nbrx.drop("MIN_SVC_DT", "LAG_SVC_DT", "CLAIM_RANK")
print('Shape :', paid_claims_w_nbrx.count(), len(paid_claims_w_nbrx.columns))

In [0]:
paid_claims_w_nbrx.display()

In [0]:
paid_claims_w_nbrx = paid_claims_w_nbrx.select(['CLAIM_ID','PROVIDER_ID','SVC_DT','NBRX_FLAG']).distinct()
print('Shape :', paid_claims_w_nbrx.count(), len(paid_claims_w_nbrx.columns))

In [0]:
paid_claims_w_nbrx = paid_claims_w_nbrx.filter(col('SVC_DT')>= data_start)
print('Shape :', paid_claims_w_nbrx.count(), len(paid_claims_w_nbrx.columns))

In [0]:
# Step 1: Keep claims in each cohort's prediction month (PREDICTION_START_DATE .. PREDICTION_END_DATE, inclusive)
coh_win = df_cohort.select("cohort_id", "PREDICTION_START_DATE", "PREDICTION_END_DATE")
df_kerendia = (
    paid_claims_w_nbrx.withColumn("SVC_DT", F.to_date("SVC_DT"))
    .join(F.broadcast(coh_win),
          (F.col("SVC_DT") >= F.col("PREDICTION_START_DATE")) & (F.col("SVC_DT") <= F.col("PREDICTION_END_DATE")))
    .select("PROVIDER_ID", "CLAIM_ID", "SVC_DT", "NBRX_FLAG", "cohort_id")
)
print('Shape :', df_kerendia.count(), len(df_kerendia.columns))

# Step 2: Aggregate NBRx per HCP x cohort
df_kerendia_agg = (
    df_kerendia.groupBy("PROVIDER_ID", "cohort_id")
    .agg(F.sum("NBRX_FLAG").alias(TARGET_COL))
)
print('Shape :', df_kerendia_agg.count(), len(df_kerendia_agg.columns))

# Step 3: Every NBA 3.1 universe HCP x every claims cohort (c1..c28)
provider_cohort_grid = hcp_universe.crossJoin(F.broadcast(df_cohort))
print('Shape :', provider_cohort_grid.count(), len(provider_cohort_grid.columns))

# Step 4: Left join and fill missing with 0; keys aligned with the 3.1 HCP feature outputs
df_kerendia_final = (
    provider_cohort_grid
    .join(df_kerendia_agg, on=["PROVIDER_ID", "cohort_id"], how="left")
    .fillna(0, subset=[TARGET_COL])
    .withColumn("SVC_DT_M", F.col("LOOKBACK_END_M"))
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "cohort_id"))
    .withColumn("PULL_DT", F.lit(dt.today().strftime("%Y_%m_%d")))
    .select("PROVIDER_ID", "cohort_id", "dataset", "SVC_DT_M", "PREDICTION_START_DATE", "PREDICTION_END_DATE",
            "HCP_COHORT_ID", F.col(TARGET_COL).cast("int").alias(TARGET_COL), "PULL_DT")
)

print('Shape :', df_kerendia_final.count(), len(df_kerendia_final.columns))
df_kerendia_final.display()

In [0]:
# Grid completeness, key uniqueness, universe membership, and NBRx totals reconcile with the windowed claims
n_rows = df_kerendia_final.count()
assert n_rows == n_hcp * len(cohorts), (n_rows, n_hcp * len(cohorts))
assert df_kerendia_final.select("HCP_COHORT_ID").distinct().count() == n_rows, "HCP_COHORT_ID not unique"
assert df_kerendia_final.filter(F.col(TARGET_COL) < 0).count() == 0
in_univ = df_kerendia.join(hcp_universe, "PROVIDER_ID", "left_semi").agg(F.sum("NBRX_FLAG")).collect()[0][0] or 0
assert df_kerendia_final.agg(F.sum(TARGET_COL)).collect()[0][0] == in_univ

(df_kerendia_final.groupBy("cohort_id", "dataset", "PREDICTION_START_DATE")
   .agg(F.sum(TARGET_COL).alias("TOTAL_NBRX"),
        F.sum((F.col(TARGET_COL) > 0).cast("int")).alias("HCPS_WITH_NBRX"),
        F.max(TARGET_COL).alias("MAX_NBRX"))
   .withColumn("_N", F.expr("int(substr(cohort_id, 2))")).orderBy("_N").drop("_N")
   .display())

In [0]:
# Save the output file (Delta) at OUTPUT_PATH from cell 3
os.makedirs(TARGET_VARIABLE_FOLDER, exist_ok=True)
(df_kerendia_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUTPUT_PATH))
tv = spark.read.format("delta").load(OUTPUT_PATH)
print(f"Saved {tv.count():,} rows, {len(tv.columns)} columns -> {OUTPUT_PATH}")

In [0]:
print("END OF NOTEBOOK")